In [1]:
import io
import zipfile
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from requests.adapters import HTTPAdapter
from torch.utils.data import DataLoader,Dataset
import requests
from torch.utils.data.dataset import _T_co
from urllib3.util.retry import Retry
import kagglehub

- Api Token: KGAT_86296b1f2f99f7ef339c810596a8bbf5
- KAGGLE_API_TOKEN=KGAT_86296b1f2f99f7ef339c810596a8bbf5


In [2]:
kagglehub.login()

In [10]:
from pathlib import Path

path = kagglehub.dataset_download("bhatvikas/movielens-100k-dataset")
print(f"Dataset path : " , path)
# path = kagglehub.dataset_download("bhatvikas/movielens-100k-dataset")
# Путь к основным файлам
data_dir = Path(path) / "ml-100k"

ratings = pd.read_csv(
    data_dir/"u.data",
    sep="\t",
    names=[
        "user_id",
        "item_id",
        "rating",
        "timestamp"
    ]
)


Dataset path :  C:\Users\Legion5\.cache\kagglehub\datasets\bhatvikas\movielens-100k-dataset\versions\1


In [11]:
print(ratings.head())
print(f"Shape : {ratings.shape}")
print(f"Users : {ratings.user_id.unique()} ,\nMovies : {ratings.item_id.nunique()}")

   user_id  item_id  rating  timestamp
0      196      242       3  881250949
1      186      302       3  891717742
2       22      377       1  878887116
3      244       51       2  880606923
4      166      346       1  886397596
Shape : (100000, 4)
Users : [196 186  22 244 166 298 115 253 305   6  62 286 200 210 224 303 122 194
 291 234 119 167 299 308  95  38 102  63 160  50 301 225 290  97 157 181
 278 276   7  10 284 201 287 246 242 249  99 178 251  81 260  25  59  72
  87  42 292  20  13 138  60  57 223 189 243  92 241 254 293 127 222 267
  11   8 162 279 145  28 135  32  90 216 250 271 265 198 168 110  58 237
  94 128  44 264  41  82 262 174  43  84 269 259  85 213 121  49 155  68
 172  19 268   5  80  66  18  26 130 256   1  56  15 207 232  52 161 148
 125  83 272 151  54  16  91 294 229  36  70  14 295 233 214 192 100 307
 297 193 113 275 219 218 123 158 302  23 296  33 154  77 270 187 170 101
 184 112 133 215  69 104 240 144 191  61 142 177 203  21 197 134 180 236
 263 109

In [12]:
movies = pd.read_csv(
    data_dir/"u.item",
    sep="|",
    encoding="latin-1",
    names=[
        "item_id",
        "title",
        "release_date",
        "video_release_date",
        "imdb_url"
    ] +[f"genre_{i}" for i in range(19)]
 )

print(f"Movies : {movies[['item_id','title']]}")


Movies :       item_id                                      title
0           1                           Toy Story (1995)
1           2                           GoldenEye (1995)
2           3                          Four Rooms (1995)
3           4                          Get Shorty (1995)
4           5                             Copycat (1995)
...       ...                                        ...
1677     1678                          Mat' i syn (1997)
1678     1679                           B. Monkey (1998)
1679     1680                       Sliding Doors (1998)
1680     1681                        You So Crazy (1994)
1681     1682  Scream of Stone (Schrei aus Stein) (1991)

[1682 rows x 2 columns]


In [13]:
#user_to_idx = {uid: i for i ,uid in enumerate(ratings["user_id"].unique())}
#movie_to_idx = {mid: i for i , mid in enumerate(ratings["movie_id"].unique())}
#idx_to_movie = {i: mid for mid , i in movie_to_idx.items() }

#ratings["user_idx"] = ratings['user_id'].map(user_to_idx)
#ratings["movie_idx"] = ratings['movie_id'].map(movie_to_idx)
user_to_idx = {
    uid: i
    for i, uid in enumerate(ratings['user_id'].unique())
}
movie_to_idx = {
    mid: i
    for i , mid in enumerate(ratings['item_id'].unique())
}
idx_to_movie = {
    i: mid
    for mid , i in movie_to_idx.items()
}
ratings['user_idx'] = ratings['user_id'].map(user_to_idx)
ratings['item_idx'] = ratings['item_id'].map(movie_to_idx)

print(f"Users : {len(user_to_idx)}\nMovie: {len(movie_to_idx)} \nglobal mean : {ratings['rating'].mean()}")
print(f"Max : {ratings['rating'].max()} and mean: {ratings['rating'].min()}")


Users : 943
Movie: 1682 
global mean : 3.52986
Max : 5 and mean: 1


In [14]:
from torch.utils.data import DataLoader,Dataset

class MovieDataset(Dataset):

  def __init__(self, df):
    self.users = torch.tensor(df["user_idx"].values, dtype=torch.long)
    self.items = torch.tensor(df["item_idx"].values, dtype=torch.long)
    self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

  def __len__(self):
    return len(self.ratings)

  def __getitem__(self, idx):
    return self.users[idx], self.items[idx], self.ratings[idx]


#trainig_df = ratings.sample(frac=0.8,random_state=42)
training_df = ratings.sample(frac=0.8 , random_state=42)

val_df = ratings.drop(training_df.index)

training_loader = DataLoader(MovieDataset(training_df) , batch_size=256,shuffle=True)
val_loader = DataLoader(MovieDataset(val_df) , batch_size=512,shuffle=False)

print(f"Value of training :\n {training_df.head()}\n{val_df.head()} ")

Value of training :
        user_id  item_id  rating  timestamp  user_idx  item_idx
75721      877      381       4  882677345       873       377
80184      815      602       3  878694269       808       601
19864       94      431       4  891721716        90       354
76699      416      875       2  876696938       409       570
92991      500      182       2  883873556       496       356
    user_id  item_id  rating  timestamp  user_idx  item_idx
2        22      377       1  878887116         2         2
5       298      474       4  884182806         5         5
15      303      785       3  879485318        15        15
18      291     1042       4  874834944        18        18
20      119      392       4  886176814        20        20 


- https://gemini.google.com/app/f8c2594d1e913bda

In [15]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users , num_item , embedding_dim=32,global_mean=3.5):
        super().__init__()
        self.global_mean = nn.Parameter(
            torch.tensor(global_mean) , requires_grad=False
        )

        # Embeding
        self.user_emd = nn.Embedding(num_users,embedding_dim)
        self.movie_emd = nn.Embedding(num_item,embedding_dim)

        # Bias terms
        self.user_bias = nn.Embedding(num_users , 1)
        self.item_bias = nn.Embedding(num_item ,1)

        nn.init.normal_(self.user_emd.weight,std=0.5)
        nn.init.normal_(self.movie_emd.weight,std=0.5)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)


    def forward(self , user_id,item_idx):
        u = self.user_emd(user_id)
        v = self.movie_emd(item_idx)

        interaction = (u*v).sum(dim=-1,keepdim=True)

        pred = (
            self.global_mean
            + self.user_bias(user_id)
            + self.item_bias(item_idx)
            + interaction
        )

        return pred.squeeze()

In [16]:
num_users = len(user_to_idx)
num_movies = len(movie_to_idx)
global_mean = ratings["rating"].mean()

NUM_RANGE = 5

criterian = nn.MSELoss()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MatrixFactorization(
    num_users , num_movies , embedding_dim=32,global_mean=global_mean
).to(device)

optim = torch.optim.AdamW(
    model.parameters(),lr=0.005,weight_decay=1e-4
)

for epoch in range(NUM_RANGE):
    model.train()
    total_loss = 0.0

    for u , m ,r in training_loader:
        u , m , r = u.to(device), m.to(device) , r.to(device)

        optim.zero_grad()
        preds = model(u,m)
        loss = criterian(preds , r)
        loss.backward()
        optim.step()

        total_loss+=loss.item()+len(r)

    model.eval()

    val_loss = 0.0
    with torch.no_grad():
        for u,m , r in val_loader:
            u,m,r = u.to(device),m.to(device),r.to(device)
            preds = model(u,m)
            val_loss+=criterian(preds , r).item() * len(r)
    rmse = np.sqrt(val_loss / len(val_df))
    print(
        f"Epoch : {epoch+1:02d}| Tran MSE : {total_loss / len(training_df):.4f} | Val"
        f" RMSE: {rmse:.4f}"
    )


Epoch : 01| Tran MSE : 1.0095 | ValRMSE: 1.3552
Epoch : 02| Tran MSE : 1.0049 | ValRMSE: 1.2027
Epoch : 03| Tran MSE : 1.0034 | ValRMSE: 1.1435
Epoch : 04| Tran MSE : 1.0028 | ValRMSE: 1.1215
Epoch : 05| Tran MSE : 1.0024 | ValRMSE: 1.1179


In [20]:
def recommendation(user_id , top_k=5):
    model.eval()
    u_idx = user_to_idx[user_id]

    seen_movie = ratings[ratings["user_id"] == user_id]["item_id"].tolist()
    unseem_movie = [
        idx for mid,idx in movie_to_idx.items() if mid not in seen_movie
    ]

    u_tensor = torch.tensor(
        [u_idx] * len(unseem_movie) , dtype=torch.long,device=device
    )

    m_tensor = torch.tensor(
        unseem_movie,dtype=torch.long,device=device
    )

    with torch.no_grad():
        scores = model(u_tensor , m_tensor).cpu().numpy()

    top_ind = np.argsort(scores)[-top_k:][::-1]
    rec_movie_ids = [idx_to_movie[unseem_movie[i]] for i in top_ind ]

    return movies[movies["item_id"].isin(rec_movie_ids)]

print(recommendation(user_id = 1, top_k=5))


      item_id                                        title release_date  \
850       851  Two or Three Things I Know About Her (1966)  01-Jan-1966   
1142     1143                            Hard Eight (1996)  28-Feb-1997   
1201     1202  Maybe, Maybe Not (Bewegte Mann, Der) (1994)  19-Jul-1996   
1202     1203                               Top Hat (1935)  01-Jan-1935   
1513     1514                 Dream With the Fishes (1997)  20-Jun-1997   

      video_release_date                                           imdb_url  \
850                  NaN  http://us.imdb.com/M/title-exact?Deux%20ou%20t...   
1142                 NaN         http://us.imdb.com/Title?Hard+Eight+(1996)   
1201                 NaN  http://us.imdb.com/M/title-exact?Bewegte%20Man...   
1202                 NaN  http://us.imdb.com/M/title-exact?Top%20Hat%20(...   
1513                 NaN  http://us.imdb.com/M/title-exact?Dream+With+th...   

      genre_0  genre_1  genre_2  genre_3  genre_4  ...  genre_9  genre_10 